# mse-reconstruction-loss composite — cx25: MSE reconstruction loss → backward on scalar → grads land on encoder/decoder

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `mse-reconstruction-loss`, `backward-on-scalar-loss`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "mse-reconstruction-loss"
DD_ATOM_IDS = ["mse-reconstruction-loss", "backward-on-scalar-loss"]
DD_SUBTOPICS = ["Generative: MSE reconstruction loss", "PyTorch: backward()"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

An autoencoder produces a reconstruction `x_hat` from input `x`. To train it, you reduce the per-element reconstruction error to a SCALAR and call `.backward()`. Two atoms wire together every step:

1. **mse-reconstruction-loss** — `loss = F.mse_loss(x_hat, x)` (default `reduction='mean'`). MSE is the natural recon loss for continuous-valued inputs; for binary pixels you'd use BCE instead.
2. **backward-on-scalar-loss** — `.backward()` requires its anchor to be a SCALAR (0-dim tensor). `F.mse_loss(..., reduction='mean')` already returns a scalar; `reduction='none'` would return a same-shape tensor and `.backward()` would fail with `grad can be implicitly created only for scalar outputs`.

**Anatomy.**
```python
x_hat = decoder(encoder(x))          # forward.
loss = F.mse_loss(x_hat, x)          # mse-reconstruction-loss (scalar by default).
loss.backward()                       # backward-on-scalar-loss → fills .grad on all params.
```

**Why test these together.** A common student bug is `loss = (x_hat - x).pow(2)` (no reduction) followed by `.backward()` — fails because the loss is not scalar. The fix is `.mean()` (or `F.mse_loss`'s default). Both atoms have to be respected jointly.

### Composite Exercise — MSE reconstruction loss → backward on scalar → grads land on encoder/decoder

**Atoms exercised together**: `mse-reconstruction-loss`, `backward-on-scalar-loss`

Implement `cx25_ae_recon_backward(encoder, decoder, x)`.

Inputs:
- `encoder`, `decoder` — two `nn.Module` instances. Calling `decoder(encoder(x))` produces   `x_hat` with the same shape as `x`.
- `x` — an input batch tensor.

Required behaviour:
1. Compute `x_hat = decoder(encoder(x))`.
2. Compute the scalar MSE recon loss via `F.mse_loss(x_hat, x)` (atom: mse-reconstruction-loss). The result MUST be a 0-dim tensor.
3. Call `loss.backward()` (atom: backward-on-scalar-loss). This populates `.grad` on every parameter in encoder + decoder.
4. Return the scalar `loss` tensor (NOT `loss.item()` — the test inspects it).

The test verifies: the returned loss is a scalar grad-tracking tensor, `.grad` is populated on every encoder + decoder parameter (all non-None and non-zero), and the loss is non-negative.

In [ ]:
def cx25_ae_recon_backward(encoder, decoder, x):
    # Forward through encoder then decoder.
    x_hat = decoder(encoder(x))
    # Atom A (mse-reconstruction-loss): scalar mean-squared-error.
    loss = F.mse_loss(x_hat, x)
    # Atom B (backward-on-scalar-loss): backward requires scalar output.
    loss.backward()
    return loss


<details><summary>Show solution — cx25</summary>

```python
def cx25_ae_recon_backward(encoder, decoder, x):
    # Forward through encoder then decoder.
    x_hat = decoder(encoder(x))
    # Atom A (mse-reconstruction-loss): scalar mean-squared-error.
    loss = F.mse_loss(x_hat, x)
    # Atom B (backward-on-scalar-loss): backward requires scalar output.
    loss.backward()
    return loss
```

`F.mse_loss(reduction='mean')` (the default) gives a scalar — exactly what `.backward()` needs. If you used `reduction='none'` you'd have to call `.mean()` (or `.sum()`) before `.backward()`, otherwise PyTorch raises `RuntimeError: grad can be implicitly created only for scalar outputs`.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx25'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx25',
        'subtopics': ["Generative: MSE reconstruction loss", "PyTorch: backward()"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()